# Hands-on Lab: Extract, Transform and Load GDP Data

## Introduction

In this practice project, an end-to-end ETL (Extract, Transform and Load) pipeline is developed using Python. The pipeline retrieves GDP data from a public web source, transforms it into a business-ready format, stores the processed data in multiple output formats, loads it into a SQLite database, and validates the results using SQL queries.

## Project Scenario

An international company is expanding its business operations across multiple countries and requires an automated solution to monitor the latest Gross Domestic Product (GDP) rankings published by the International Monetary Fund (IMF).

As a Data Engineer, the objective is to develop an automated ETL pipeline capable of extracting GDP data, transforming the values from millions to billions of USD, storing the results in multiple formats, querying the data from a database, and logging each stage of the execution process.

## Objectives

This notebook performs the following tasks:

- Extract GDP information from the source website.
- Transform GDP values from millions to billions of USD.
- Save the transformed data as CSV and JSON files.
- Load the transformed data into a SQLite database.
- Execute a SQL query to validate the loaded data.
- Log each stage of the ETL process.

## Data Source

The GDP data is extracted from the following archived webpage:

https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29

## Task 1 – Import Required Libraries

Import the required Python libraries:

- pandas
- sqlite3
- requests
- BeautifulSoup
- datetime
- numpy

In [1]:
import pandas as pd
import sqlite3
import requests
from bs4 import BeautifulSoup
from datetime import datetime as dt
import numpy as np

## Task 2 – Configure the Project

Define the project configuration, including:

- Source URL
- DataFrame attributes
- Database information
- Output file paths used throughout the ETL pipeline

In [1]:
url = 'https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29'
table_attribs = ["Country", "GDP_USD_millions"]
db_name = 'World_Economies.db'
table_name = 'Countries_by_GDP'
csv_path = './Countries_by_GDP.csv'
json_path = './Countries_by_GDP.json'

## Task 3 – Create the ETL Functions

Create the functions required to perform each stage of the ETL pipeline, including:

- Data extraction, transformation and loading the processed data into different formats
- Executing SQL queries
- Logging the progress of the pipeline

In [3]:
def extract(url, table_attribs):
    page = requests.get(url).text
    data = BeautifulSoup(page,'html.parser')
    df = pd.DataFrame(columns=table_attribs)
    tables = data.find_all('tbody')
    rows = tables[2].find_all('tr')
    for row in rows:
        col = row.find_all('td')
        if len(col)!=0:
            if col[0].find('a') is not None and '—' not in col[2]:
                data_dict = {"Country": col[0].a.contents[0],
                             "GDP_USD_millions": col[2].contents[0]}
                df1 = pd.DataFrame(data_dict, index=[0])
                df = pd.concat([df,df1], ignore_index=True)
    return df


def transform(df):
    GDP_list = df["GDP_USD_millions"].tolist()
    GDP_list = [float("".join(x.split(','))) for x in GDP_list]
    GDP_list = [np.round(x/1000,2) for x in GDP_list]
    df["GDP_USD_millions"] = GDP_list
    df=df.rename(columns = {"GDP_USD_millions":"GDP_USD_billions"})
    return df


def load_to_csv(df, csv_path):
    df.to_csv(csv_path, index=False)


def load_to_json(df, json_path):
    df.to_json(json_path, orient='records', indent=4)


def load_to_db(df, sql_connection, table_name):
    df.to_sql(table_name, sql_connection, if_exists='replace', index=False)


def run_query(query_statement, sql_connection):
    print(query_statement)
    query_output = pd.read_sql(query_statement, sql_connection)
    print(query_output)


def log_progress(message): 
    timestamp_format = '%d/%m/%Y %H:%M:%S' # Day/Month/Year Hour-Minute-Second 
    now = dt.now() # get current timestamp 
    timestamp = now.strftime(timestamp_format) 
    with open("./etl_project_log.txt","a") as f: 
        f.write(timestamp + ' : ' + message + '\n')

## Task 4 – Execute the ETL Pipeline

Execute the ETL pipeline by:

- Extracting the GDP data from the source website
- Transforming the values into billions of USD
- Saving the processed data as CSV and JSON files
- Loading the data into a SQLite database
- Running a SQL validation query
- Recording the progress of each stage in the execution log

In [4]:
with open("./etl_project_log.txt", "w"):
    pass

log_progress('Preliminaries complete. Initiating ETL process')

df = extract(url, table_attribs)

log_progress('Data extraction complete. Initiating Transformation process')

df = transform(df)

log_progress('Data transformation complete. Initiating loading process')

load_to_csv(df, csv_path)

log_progress('Data saved to CSV file')

load_to_json(df, json_path)

log_progress('Data saved to JSON file')

sql_connection = sqlite3.connect(db_name)

log_progress('SQL Connection initiated.')

load_to_db(df, sql_connection, table_name)

log_progress('Data loaded to Database as table. Running the query')

query_statement = f"SELECT * FROM {table_name} WHERE GDP_USD_billions >= 100"
run_query(query_statement, sql_connection)

log_progress('Process Complete.')

sql_connection.close()

SELECT * FROM Countries_by_GDP WHERE GDP_USD_billions >= 100
          Country  GDP_USD_billions
0   United States          26854.60
1           China          19373.59
2           Japan           4409.74
3         Germany           4308.85
4           India           3736.88
..            ...               ...
64          Kenya            118.13
65         Angola            117.88
66           Oman            104.90
67      Guatemala            102.31
68       Bulgaria            100.64

[69 rows x 2 columns]


## Task 5 – Review the Results

Verify that the ETL pipeline completed successfully by reviewing:

- The generated output files
- SQL query results
- Execution log.

In [5]:
# Display the first five rows of the transformed DataFrame
df.head()

,Country,GDP_USD_billions
0,United States,26854.60
1,China,19373.59
2,Japan,4409.74
3,Germany,4308.85
4,India,3736.88


In [6]:
# Display the execution log
with open("etl_project_log.txt", "r") as f:
    print(f.read())

30/06/2026 20:34:31 : Preliminaries complete. Initiating ETL process
30/06/2026 20:34:34 : Data extraction complete. Initiating Transformation process
30/06/2026 20:34:34 : Data transformation complete. Initiating loading process
30/06/2026 20:34:34 : Data saved to CSV file
30/06/2026 20:34:34 : Data saved to JSON file
30/06/2026 20:34:34 : SQL Connection initiated.
30/06/2026 20:34:34 : Data loaded to Database as table. Running the query
30/06/2026 20:34:34 : Process Complete.

